In [1]:
# ==============================================================================
# GLOBAL DEPENDENCIES & PIPELINE INITIALIZATION
# ==============================================================================
import os
import pickle
import numpy as np
import pandas as pd

# Core SciPy sparse matrix tools
from scipy.sparse import csr_matrix, hstack, save_npz, load_npz

# Visual analytics suite
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import normalize

# Ensure the data architecture directory exists globally
os.makedirs('data/features', exist_ok=True)

print("🚀 Master feature environment and sparse data paths initialized successfully.")

🚀 Master feature environment and sparse data paths initialized successfully.


In [3]:
# 1. Load the row index mappings
with open('../data/features/album_ids.pkl', 'rb') as f:
    album_id_order = pickle.load(f)

# 2. Load all feature blocks from the folder
X_tags    = load_npz('../data/features/album_tags_matrix.npz')
X_labels  = load_npz('../data/features/album_labels_matrix.npz')
X_types   = load_npz('../data/features/album_types_matrix.npz')
X_ratings = load_npz('../data/features/album_ratings_matrix.npz')

# 3. Align all matrices to the full album universe.
#    album_ids.pkl may have been built from a filtered subset (e.g. only albums
#    with tags), so we expand each matrix to cover every album in mb_album.parquet,
#    inserting zero rows for albums that had no tags/labels/ratings.
full_album_ids = pd.Index(
    pd.read_parquet('../data/mb_album.parquet', columns=['id'])['id'].sort_values()
)

if len(album_id_order) < len(full_album_ids):
    print(f"Expanding matrices from {len(album_id_order):,} → {len(full_album_ids):,} albums...")
    current_pos = full_album_ids.get_indexer(album_id_order)
    n_full = len(full_album_ids)

    def _expand(X, row_pos, n):
        coo = X.tocoo()
        return csr_matrix((coo.data, (row_pos[coo.row], coo.col)), shape=(n, X.shape[1]))

    X_tags    = _expand(X_tags,    current_pos, n_full)
    X_labels  = _expand(X_labels,  current_pos, n_full)
    X_types   = _expand(X_types,   current_pos, n_full)
    X_ratings = _expand(X_ratings, current_pos, n_full)
    album_id_order = full_album_ids.tolist()

# 4. Combine horizontally into the complete feature matrix
X_final_album_knn = hstack([X_tags, X_labels, X_types, X_ratings]).tocsr()

print(f"🚀 Matrix built out of data/features/ successfully!")
print(f"Final Model Dimensions: {X_final_album_knn.shape[0]:,} albums x {X_final_album_knn.shape[1]:,} features")

Expanding matrices from 1,008,102 → 2,241,402 albums...
🚀 Matrix built out of data/features/ successfully!
Final Model Dimensions: 2,241,402 albums x 6,521 features


In [4]:
import os

print(f"Working directory: {os.getcwd()}")
print()

matrices = {
    "X_tags":             X_tags,
    "X_labels":           X_labels,
    "X_types":            X_types,
    "X_ratings":          X_ratings,
    "X_final_album_knn":  X_final_album_knn,
}

for name, X in matrices.items():
    print(f"{name:25s}  shape={str(X.shape):25s}  nnz={X.nnz:,}")

print()
print(f"album_id_order length: {len(album_id_order):,}")

Working directory: /Users/niall/Desktop/ai_eng/mixtape/mixtape/features

X_tags                     shape=(2241402, 3041)            nnz=3,004,997
X_labels                   shape=(2241402, 3469)            nnz=402,047
X_types                    shape=(2241402, 10)              nnz=402,047
X_ratings                  shape=(2241402, 1)               nnz=44,334
X_final_album_knn          shape=(2241402, 6521)            nnz=3,853,425

album_id_order length: 2,241,402
